<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 解读一份 trace:14 条消息背后的三件套协作

**一句话定位**:从一个真实的 MCP 研究任务,看 deep agent 如何调度 **TODO、Files、Subagents** 完成工作。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**任务**:用户问 `Give me an overview of Model Context Protocol (MCP).`

**Agent 类型**:Lesson 3 的 orchestrator agent(带 `task` 工具,可派单给 `research-agent`)

**消息总数**:14 条 = 1 Human + 6 AI + 7 Tool

**用到的工具**:`write_todos`、`ls`、`write_file`、`task`、`think_tool`、`read_todos`

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📋 鸟瞰:14 条消息一表掌握

</div>

| # | 类型 | 调用 / 内容 | 阶段 |
|---|------|-------------|------|
| 1 | Human | `Give me an overview of MCP.` | 提问 |
| 2 | AI | `write_todos(2 项)` + `ls()` | 🧩 规划 |
| 3 | Tool | write_todos 成功 | — |
| 4 | Tool | ls 返回 `[]`(空) | — |
| 5 | AI | `write_file(user_request.md)` + `task(...)` | 🚀 派单 |
| 6 | Tool | write_file 成功 | — |
| 7 | Tool | **task 返回 ~3000 字 MCP 综述** | — |
| 8 | AI | `think_tool(reflection=...)` | 🔄 复盘 |
| 9 | Tool | reflection 已记录 | — |
| 10 | AI | `read_todos()` | 🔄 巡检 |
| 11 | Tool | 当前 TODO 列表 | — |
| 12 | AI | `write_todos(2 项 → completed)` | 🔄 收尾 |
| 13 | Tool | TODO 更新成功 | — |
| 14 | AI | **给用户的最终 MCP 综述** | 📝 答复 |

**节奏特征**:6 条 AI 消息里有 5 条带 `tool_use`,**只有最后 1 条** 是纯文本(`stop_reason = end_turn`)。这就是 ReAct 循环的典型形态——「思考 → 动手 → 看结果 → 再思考」,直到 agent 认为可以收尾。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 三件套在这次任务里的分工

</div>

| 工具 | 治什么病 | 在本 trace 里做了什么 | 出现位置 |
|------|---------|----------------------|----------|
| **write_todos / read_todos** | 治忘(规划丢失) | 创建 2 项 → 巡检 → 标记 `completed` | #2、#10、#12 |
| **write_file** | 治挤(上下文爆炸) | 把用户原话落盘到 `user_request.md` | #5 |
| **task** | 治乱(注意力涣散) | 派单给 research-agent,**隔离 context** | #5 |
| **think_tool** | 复述(recitation) | 反思研究结果是否够用 | #8 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:#5 这一条 AIMessage 同时调了 `write_file` + `task`,这是 **并行工具调用**——一次 AIMessage 的 `tool_calls` 数组可以挂多个 tool_call。LangGraph 的 `ToolNode` 会并发执行它们,然后把每个 ToolMessage 依次回填进 `state["messages"]`。所以 #6 和 #7 才会紧挨着出现。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧩 阶段一:规划(#1 → #4)

</div>

**Agent 的第一反应**:不是直奔答案,而是 **先规划 + 先盘点**。

第 2 条 AIMessage 的 `tool_calls`:

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">tool_calls = [
    {'name': 'write_todos', 'args': {'todos': [
        {'content': 'Delegate research on MCP to sub-agent',
         'status': '<span style="background:#3d3414; color:#fde047; padding:0 2px;">in_progress</span>'},
        {'content': 'Review research findings and compile overview',
         'status': 'pending'}
    ]}},
    {'name': 'ls', 'args': {}}   # ← 盘点:之前有没有相关文件?
]
</code></pre>

两件事并行做完——任务清单先立,工作目录先扫。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:为什么开局先 `ls()`?

不是为了「找答案」,是为了 **确认 context 状态**——这台 agent 可能在长 session 里被多次调用,虚拟文件系统里可能已有上下文(之前的研究、半成品、用户偏好)。

ls 返回 `[]` 之后,agent 才安心 **从零开始**。这一步在长 session 里非常关键。短 session 看起来「多余」,其实是**良好习惯**——agent 没有假设 context 是空的。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🚀 阶段二:派单(#5 → #7)

</div>

**核心动作**:把任务 **外包** 出去,自己只等结果。

第 5 条 AIMessage 一次性调了 2 个工具:

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">tool_calls = [
    # 1) 把用户原话落盘,以防主 context 滚动后丢失
    {'name': 'write_file', 'args': {
        'file_path': 'user_request.md',
        'content': '# User Request\n\nGive me an overview of MCP.\n'
    }},
    # 2) 派单给 research-agent
    {'name': '<span style="background:#3d3414; color:#fde047; padding:0 2px;">task</span>', 'args': {
        'subagent_type': 'research-agent',
        'description': 'Research the MCP. I need: what MCP is,
                        purpose, key features, how it works, ...'
    }}
]
</code></pre>

ToolMessage #7 返回了 research-agent 的研究结果——大约 **3000 字**,涵盖 MCP 的定义、开发者、目的、特性、架构、用例、意义。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:`task` 工具的 `description` 是 **主 agent 写给子 agent 的「需求说明书」**。

主 agent 自己的 context 里此刻已有 6 条消息(user 原话、write_todos 结果、ls 结果……),但子 agent **完全看不到这些**——只看到这一段精心写好的 description。

这就是 lesson 3 主课讲的 **context isolation**:`task` 工具内部一句 `state["messages"] = [{"role":"user","content":description}]`,把父消息列表整个换掉。

📎 详见 [3_c_⭐️_subagent_state隔离.ipynb](./3_c_⭐️_subagent_state隔离.ipynb)

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔄 阶段三:复盘 & 收尾(#8 → #13)

</div>

拿到 research-agent 的 3000 字回答后,主 agent **没有直接答用户**,而是走了 3 步「内务」:

| 步骤 | 工具 | 目的 |
|------|------|------|
| #8 | `think_tool(reflection=...)` | 自言自语:研究够不够全?有没有遗漏? |
| #10 | `read_todos()` | 复述 TODO,把注意力拉回原计划 |
| #12 | `write_todos(... → completed)` | 标记完成,清出脑容量 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:这就是 **recitation(复述)**——Manus 团队提出的术语。

Agent 通过 **不断「写下来再读一遍」自己的 TODO 和反思**,把注意力反复拉回主线。`think_tool` 和 `read_todos` 都属于 **self-prompt tool**——它们的 ToolMessage 内容会回流进 context,让 LLM 在下一轮看到自己刚才在想什么。

</div>

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**反例:如果没有这三件套,会怎样?**

- ❌ 没有 **TODO**:agent 看到 3000 字研究结果太长,**忘了**还要 "compile overview",可能直接跳过收尾
- ❌ 没有 **Files**:用户原话在 context 滚动后被淹没,**主线漂移**(开始答 "What is OAuth 2.1?" 之类的子话题)
- ❌ 没有 **task**:research-agent 直接跑在主 context 里,3000 字结果**永久塞进** 后续每一轮 LLM 调用,**注意力被污染**
- ❌ 没有 **think_tool**:拿到结果就盲目答,**质量失控**

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📝 阶段四:最终答案(#14)

</div>

第 14 条 AIMessage **没有 `tool_calls`,没有 thinking**,只有 `content` 字符串——这就是 ReAct 循环的终点。

`stop_reason` 从前面的 `tool_use` 变成 `end_turn`,标志着 LLM 认为「**任务完成,无需再调工具**」。

内容是基于子 agent 返回的 3000 字研究结果 **改写** 成的简洁版(682 output tokens),不是原文复制。主 agent 起了 **润色 + 控篇幅 + 调整结构** 的作用——这正是 orchestrator 的核心职责:**不亲自干活,只整合产出**。

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**看 trace 不要看消息,要看阶段** —— 规划 → 派单 → 复盘 → 答复。这四步是 deep agent 的固定节拍,**三件套就是支撑这四步的基础设施**。

</div>